# PlantDoc AI — Advanced Fine-Tuning dan Lokalisasi Grad-CAM

Notebook ini melatih classifier **MobileNetV2 lima kelas** untuk daun tomat: `healthy`, `early`, `late`, `mold`, dan `unknown`. Setelah evaluasi held-out, notebook membuat **heatmap Grad-CAM dan kotak perkiraan area respons model** pada gambar yang diprediksi sakit.

**Penting:** kotak Grad-CAM adalah visualisasi explainability, bukan ground-truth penyakit dan bukan model object detection. Karena dataset tidak memiliki anotasi kotak/segmentasi lesi, notebook tidak menghitung IoU atau mAP. Kotak hanya dibuat untuk prediksi penyakit yang melewati ambang keyakinan; daun sehat dan `unknown` tidak diberi kotak penyakit.

**Sebelum mulai:**
1. Pilih runtime Google Colab T4 GPU.
2. Jalankan sel secara berurutan dari atas.
3. Untuk Kaggle, simpan `KAGGLE_USERNAME` dan `KAGGLE_KEY` di Colab Secrets; jangan menulis token ke notebook atau Git.
4. Jika Colab meminta restart setelah instalasi paket, restart runtime lalu lanjutkan dari sel berikutnya.
5. Simpan artefak ke Google Drive setelah semua sel selesai.

**Yang ditingkatkan:** provenance dan deduplikasi data, group-aware split, kelas `unknown`, class weight, label smoothing, two-stage fine-tuning, pemilihan checkpoint berdasarkan `val_loss`, temperature scaling dari validation set, perbandingan baseline head-only versus partial fine-tuning, metrik unknown dan kalibrasi, confusion matrix, reliability diagram, Grad-CAM, kotak area respons, ekspor gambar contoh, serta validasi paritas Keras–LiteRT.


In [ ]:
!nvidia-smi
!pip -q install "tensorflow==2.21.0" "keras==3.15.1" "scikit-learn==1.9.1" "matplotlib==3.11.2" "kagglehub==1.0.3" "datasets==5.2.0" "tensorflow-datasets==4.9.10" "pillow==13.0.0" "numpy==2.5.5" "scipy==1.18.1" "ai-edge-litert==2.4.0"


## 1. Dataset, provenance, deduplikasi, dan split

Sumber daun dicoba berurutan: Kaggle, Hugging Face, lalu TensorFlow Datasets. Setiap percobaan dimulai dari direktori bersih. Gambar duplikat identik dibuang berdasarkan SHA-256. Split 70/15/15 dilakukan per kelas dengan `GroupShuffleSplit`; group berasal dari nama file sumber atau ID baris dataset.

Kelas `unknown` mencakup tanaman di luar cakupan dan objek non-tanaman CIFAR-10. Ini meningkatkan rejection berdasarkan data training, tetapi **bukan jaminan open-set rejection** untuk semua gambar baru.

Output tahap ini:
- `dataset_manifest.json`: sumber, revisi, jumlah kelas, duplikat, dan jumlah tiap split.
- `split_manifest.csv`: asal setiap gambar, group, hash, kelas, dan split.


In [ ]:
import csv
import hashlib
import json
import platform
import shutil
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import kagglehub
import tensorflow_datasets as tfds
from datasets import load_dataset
from google.colab import userdata
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
TOMATO_SELECTED = {
    "healthy": "Tomato___healthy",
    "early": "Tomato___Early_blight",
    "late": "Tomato___Late_blight",
    "mold": "Tomato___Leaf_Mold",
}
SELECTED = {**TOMATO_SELECTED, "unknown": None}
HF_LABELS = {
    "healthy": "Tomato healthy",
    "early": "Tomato Early blight",
    "late": "Tomato Late blight",
    "mold": "Tomato Leaf Mold",
}
CIFAR_DATASET_ID = "uoft-cs/cifar10"
UNKNOWN_PLANT_PER_CLASS = 150
UNKNOWN_CIFAR_PER_CLASS = 600
ROOT = Path("/content/plantdoc_data")


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

sources = {key: [] for key in SELECTED}
seen_hashes = {}
stats = {key: {"loaded": 0, "duplicates": 0} for key in SELECTED}


def reset_sources():
    global sources, seen_hashes, stats
    shutil.rmtree(ROOT / "all", ignore_errors=True)
    sources = {key: [] for key in SELECTED}
    seen_hashes = {}
    stats = {key: {"loaded": 0, "duplicates": 0} for key in SELECTED}


def next_path(web_key, suffix):
    path = ROOT / "all" / web_key / f"{web_key}_{len(sources[web_key]):06d}{suffix.lower()}"
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def register_file(path, web_key, group_id, origin):
    path = Path(path)
    digest = sha256_file(path)
    if digest in seen_hashes:
        if seen_hashes[digest] != web_key:
            raise ValueError(f"Hash identik dengan label berbeda: {path}")
        stats[web_key]["duplicates"] += 1
        path.unlink(missing_ok=True)
        return
    seen_hashes[digest] = web_key
    sources[web_key].append({
        "path": path,
        "group_id": str(group_id),
        "sha256": digest,
        "origin": origin,
    })
    stats[web_key]["loaded"] += 1


def copy_source(src, web_key, group_id, origin):
    dst = next_path(web_key, Path(src).suffix)
    shutil.copy2(src, dst)
    register_file(dst, web_key, group_id, origin)


def save_image(image, web_key, group_id, origin):
    dst = next_path(web_key, ".jpg")
    image.convert("RGB").save(dst, format="JPEG", quality=95)
    register_file(dst, web_key, group_id, origin)


In [ ]:
def load_kaggle():
    username = userdata.get("KAGGLE_USERNAME")
    kaggle_key = userdata.get("KAGGLE_KEY")
    if not username or not kaggle_key:
        raise RuntimeError("Kaggle secret belum tersedia")
    dataset_id = "soumiknafiul/plantvillage-dataset-labeled"
    base = Path(kagglehub.dataset_download(dataset_id))
    variants = {
        "healthy": [("tomato", "healthy")],
        "early": [("tomato", "early", "blight")],
        "late": [("tomato", "late", "blight")],
        "mold": [("tomato", "leaf", "mold")],
    }
    extensions = {".jpg", ".jpeg", ".png"}
    for folder in base.rglob("*"):
        if not folder.is_dir():
            continue
        name = folder.name.lower()
        if "___" not in name and "background" not in name:
            continue
        web_key = next(
            (key for key, parts in variants.items() if all(part in name for part in parts)),
            None,
        )
        cap = None
        if web_key is None:
            web_key = "unknown"
            origin = "plantvillage-other"
            cap = UNKNOWN_PLANT_PER_CLASS
        else:
            origin = "plantvillage-tomato"
        saved = 0
        for src in folder.rglob("*"):
            if not src.is_file() or src.suffix.lower() not in extensions:
                continue
            copy_source(src, web_key, f"{folder.name}:{src.stem}", origin)
            saved += 1
            if cap is not None and saved >= cap:
                break
    return dataset_id, None


def load_huggingface():
    dataset_id = "manoela/plantvillage"
    dataset = load_dataset(dataset_id, split="train", streaming=True)
    caption_to_key = {label: key for key, label in HF_LABELS.items()}
    unknown_per_caption = {}
    for index, row in enumerate(dataset):
        caption = row["caption"]
        web_key = caption_to_key.get(caption)
        if web_key is not None:
            save_image(row["image"], web_key, f"hf-{index}", "plantvillage-tomato")
            continue
        if unknown_per_caption.get(caption, 0) >= UNKNOWN_PLANT_PER_CLASS:
            continue
        save_image(row["image"], "unknown", f"hf-{index}", "plantvillage-other")
        unknown_per_caption[caption] = unknown_per_caption.get(caption, 0) + 1
    revision = getattr(getattr(dataset, "info", None), "sha", None)
    return dataset_id, revision


def load_tfds():
    dataset_id = "tensorflow_datasets/plant_village"
    dataset, info = tfds.load("plant_village", split="train", as_supervised=True, with_info=True)
    class_names_full = info.features["label"].names
    selected_ids = {class_names_full.index(name): key for key, name in TOMATO_SELECTED.items()}
    unknown_per_label = {}
    for index, (image, label_id) in enumerate(tfds.as_numpy(dataset)):
        label_index = int(label_id)
        web_key = selected_ids.get(label_index)
        if web_key is not None:
            save_image(Image.fromarray(image), web_key, f"tfds-{index}", "plantvillage-tomato")
            continue
        if unknown_per_label.get(label_index, 0) >= UNKNOWN_PLANT_PER_CLASS:
            continue
        save_image(Image.fromarray(image), "unknown", f"tfds-{index}", "plantvillage-other")
        unknown_per_label[label_index] = unknown_per_label.get(label_index, 0) + 1
    return dataset_id, str(info.version)


def load_cifar_unknown():
    dataset = load_dataset(CIFAR_DATASET_ID, split="train", streaming=True)
    label_names = dataset.features["label"].names
    per_label = {index: 0 for index in range(len(label_names))}
    for index, row in enumerate(dataset):
        label_index = int(row["label"])
        if per_label[label_index] >= UNKNOWN_CIFAR_PER_CLASS:
            continue
        save_image(row["img"], "unknown", f"cifar-{index}", "cifar10")
        per_label[label_index] += 1
        if all(count >= UNKNOWN_CIFAR_PER_CLASS for count in per_label.values()):
            break
    missing = [label_names[index] for index, count in per_label.items() if count == 0]
    if missing:
        raise RuntimeError(f"Kelas CIFAR-10 tidak ditemukan: {missing}")
    revision = getattr(getattr(dataset, "info", None), "sha", None)
    return CIFAR_DATASET_ID, revision, len(label_names)


In [ ]:
loaders = [
    ("Kaggle", load_kaggle),
    ("Hugging Face", load_huggingface),
    ("TensorFlow Datasets", load_tfds),
]
fallback_errors = []
for source_name, loader in loaders:
    reset_sources()
    try:
        source_dataset, source_revision = loader()
        missing = [key for key in TOMATO_SELECTED if not sources[key]]
        if missing:
            raise RuntimeError(f"Kelas tomat tidak ditemukan: {missing}")
        SOURCE = source_name
        SOURCE_DATASET = source_dataset
        SOURCE_REVISION = source_revision
        break
    except Exception as error:
        fallback_errors.append({"source": source_name, "error": str(error)})
        print(f"Fallback setelah {source_name}: {error}")
else:
    raise RuntimeError("Semua sumber dataset gagal; lihat fallback_errors")

try:
    cifar_dataset, cifar_revision, cifar_class_count = load_cifar_unknown()
except Exception as error:
    raise RuntimeError("Sumber non-tanaman untuk kelas unknown gagal dimuat") from error

unknown_origin_counts = {}
for record in sources["unknown"]:
    origin = record["origin"]
    unknown_origin_counts[origin] = unknown_origin_counts.get(origin, 0) + 1
if unknown_origin_counts.get("cifar10", 0) < cifar_class_count * UNKNOWN_CIFAR_PER_CLASS:
    raise RuntimeError("Contoh CIFAR-10 untuk kelas unknown belum lengkap")

unknown_sources = [
    {
        "origin": "plantvillage-other",
        "dataset_id": SOURCE_DATASET,
        "revision": SOURCE_REVISION,
    },
    {
        "origin": "cifar10",
        "dataset_id": cifar_dataset,
        "revision": cifar_revision,
    },
]
missing = [key for key in SELECTED if not sources[key]]
if missing:
    raise RuntimeError(f"Kelas tidak ditemukan: {missing}")

split_counts = {}
split_rows = []
for web_key, records in sources.items():
    if len(records) < 6:
        raise RuntimeError(f"Kelas {web_key} hanya memiliki {len(records)} citra unik")
    paths = [record["path"] for record in records]
    groups = [record["group_id"] for record in records]
    labels = [web_key] * len(records)
    train_idx, rest_idx = next(
        GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED).split(paths, labels, groups)
    )
    rest_paths = [paths[index] for index in rest_idx]
    rest_groups = [groups[index] for index in rest_idx]
    rest_labels = [labels[index] for index in rest_idx]
    val_idx, test_idx = next(
        GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=SEED + 1).split(
            rest_paths, rest_labels, rest_groups
        )
    )
    splits = {
        "train": [records[index] for index in train_idx],
        "val": [records[rest_idx[index]] for index in val_idx],
        "test": [records[rest_idx[index]] for index in test_idx],
    }
    group_sets = {
        split: {record["group_id"] for record in split_records}
        for split, split_records in splits.items()
    }
    assert group_sets["train"].isdisjoint(group_sets["val"])
    assert group_sets["train"].isdisjoint(group_sets["test"])
    assert group_sets["val"].isdisjoint(group_sets["test"])
    split_counts[web_key] = {}
    for split, split_records in splits.items():
        destination = ROOT / split / web_key
        destination.mkdir(parents=True, exist_ok=True)
        for record in split_records:
            src = record["path"]
            output_path = destination / src.name
            shutil.copy2(src, output_path)
            split_rows.append({
                "split": split,
                "class": web_key,
                "origin": record["origin"],
                "group_id": record["group_id"],
                "sha256": record["sha256"],
                "source_path": str(src.relative_to(ROOT)),
                "path": str(output_path.relative_to(ROOT)),
            })
        split_counts[web_key][split] = len(split_records)

with (ROOT / "split_manifest.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["split", "class", "origin", "group_id", "sha256", "source_path", "path"],
    )
    writer.writeheader()
    writer.writerows(split_rows)

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "source": SOURCE,
    "dataset_id": SOURCE_DATASET,
    "revision": SOURCE_REVISION,
    "selected_classes": {
        **TOMATO_SELECTED,
        "unknown": "tanaman di luar cakupan dan objek non-tanaman",
    },
    "unknown_sources": unknown_sources,
    "unknown_sampling": {
        "plantvillage_other_per_class": UNKNOWN_PLANT_PER_CLASS,
        "cifar10_per_class": UNKNOWN_CIFAR_PER_CLASS,
    },
    "unknown_origin_counts": unknown_origin_counts,
    "deduplication": "sha256",
    "split_strategy": "group-aware per class, target 70/15/15",
    "class_counts": split_counts,
    "duplicates_removed": {key: stats[key]["duplicates"] for key in SELECTED},
    "fallback_errors": fallback_errors,
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "tensorflow": package_version("tensorflow"),
        "keras": package_version("keras"),
        "scikit_learn": package_version("scikit-learn"),
        "matplotlib": package_version("matplotlib"),
        "kagglehub": package_version("kagglehub"),
        "datasets": package_version("datasets"),
        "tensorflow_datasets": package_version("tensorflow-datasets"),
        "pillow": package_version("pillow"),
        "numpy": package_version("numpy"),
        "ai_edge_litert": package_version("ai-edge-litert"),
    },
}
(ROOT / "dataset_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("Sumber dataset:", SOURCE, SOURCE_DATASET, SOURCE_REVISION)
print("Sumber unknown:", unknown_sources)
print("Jumlah unknown per origin:", unknown_origin_counts)
for web_key in SELECTED:
    print(
        f"{web_key:8s} train={split_counts[web_key]['train']} "
        f"val={split_counts[web_key]['val']} test={split_counts[web_key]['test']} "
        f"duplikat={stats[web_key]['duplicates']}"
    )


## 2. Verifikasi integritas dataset

Pemeriksaan ini memastikan tidak ada hash duplikat setelah split, tidak ada group yang muncul di lebih dari satu split, setiap kelas terisi, dan contoh non-tanaman CIFAR-10 benar-benar masuk ke kelas `unknown`.


In [ ]:
for split in ("train", "val", "test"):
    counts = {key: len(list((ROOT / split / key).glob("*"))) for key in SELECTED}
    assert all(counts.values()), f"Split {split} memiliki kelas kosong: {counts}"
    print(f"{split:5s}: {counts}")

with (ROOT / "split_manifest.csv").open(encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))
assert len({row["sha256"] for row in rows}) == len(rows), "Ada hash duplikat setelah split"
assert any(
    row["class"] == "unknown" and row["origin"] == "cifar10"
    for row in rows
), "Kelas unknown tidak mengandung contoh non-tanaman CIFAR-10"
for web_key in SELECTED:
    groups_by_split = {
        split: {row["group_id"] for row in rows if row["class"] == web_key and row["split"] == split}
        for split in ("train", "val", "test")
    }
    assert groups_by_split["train"].isdisjoint(groups_by_split["val"])
    assert groups_by_split["train"].isdisjoint(groups_by_split["test"])
    assert groups_by_split["val"].isdisjoint(groups_by_split["test"])
print("Verifikasi deduplikasi, group split, dan sumber unknown: OK")


## 3. Pratinjau data training

Pratinjau hanya membaca data training. Data validation dan test tidak digunakan untuk memilih gambar, mengubah model, atau menyetel parameter.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

preview_classes = sorted(
    path.name for path in (ROOT / "train").iterdir() if path.is_dir()
)
fig, axes = plt.subplots(len(preview_classes), 3, figsize=(10, 2.4 * len(preview_classes)))
for row, class_name in enumerate(preview_classes):
    paths = sorted((ROOT / "train" / class_name).glob("*"))[:3]
    for col, path in enumerate(paths):
        with Image.open(path) as image:
            axes[row, col].imshow(image.convert("RGB"))
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(class_name, loc="left", fontsize=10)
fig.suptitle("Contoh data training per kelas")
fig.tight_layout()
plt.show()


## 4. Pipeline data dan class weight

Input 224×224 dan `mobilenet_v2.preprocess_input` dipertahankan agar preprocessing konsisten dengan web. Augmentasi ringan diterapkan hanya saat training. Class weight dibatasi maksimum 5.0 agar kelas minoritas tidak memberi bobot ekstrem.

Hyperparameter berada di sel ini agar eksperimen mudah diulang. Seed yang sama dipakai untuk split, augmentasi, dan training.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS_HEAD = 10
EPOCHS_FT = 12
FINE_TUNE_LAYERS = 40
LABEL_SMOOTHING = 0.05
LOCALIZATION_SAMPLES_PER_CLASS = 8
LOCALIZATION_MIN_CONFIDENCE = 0.55
CAM_PERCENTILE = 80
CAM_MIN_REGION_FRACTION = 0.005
CAM_PADDING = 0.12

train_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT / "train", image_size=IMG_SIZE, batch_size=BATCH, shuffle=True, seed=42
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT / "val", image_size=IMG_SIZE, batch_size=BATCH, shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT / "test", image_size=IMG_SIZE, batch_size=BATCH, shuffle=False
)

class_names = train_ds.class_names
print("Urutan kelas model:", class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


def infer_web_key(folder):
    name = folder.lower()
    if "healthy" in name:
        return "healthy"
    if "early" in name:
        return "early"
    if "mold" in name:
        return "mold"
    if "late" in name:
        return "late"
    return folder


web_keys = [infer_web_key(name) for name in class_names]
assert set(web_keys) == set(SELECTED), f"Peta kelas web tidak lengkap: {web_keys}"
train_counts = {
    index: len(list((ROOT / "train" / name).glob("*")))
    for index, name in enumerate(class_names)
}
total_train = sum(train_counts.values())
class_weight = {
    index: min(5.0, total_train / (len(class_names) * count))
    for index, count in train_counts.items()
}
print("Peta ke web:", dict(zip(class_names, web_keys)))
print("Jumlah train:", train_counts)
print("Class weight:", class_weight)


## 5. Model MobileNetV2 dan two-stage fine-tuning

Backbone ImageNet dibekukan pada tahap awal. Head dilatih lebih dahulu, kemudian 40 lapisan terakhir backbone dibuka dengan learning rate kecil. `training=False` pada backbone menjaga statistik BatchNormalization tidak berubah selama fine-tuning.

Label smoothing 0.05 mengurangi overconfidence. Model tetap memiliki satu input dan satu output softmax sehingga `model.tflite` tetap kompatibel dengan backend LiteRT yang ada.


In [ ]:
augment = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.06),
        layers.RandomZoom(0.08),
    ],
    name="augment",
)

backbone = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
    name="mobilenetv2",
)
backbone.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3), name="image")
x = augment(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
feature_map = backbone(x, training=False)
x = layers.GlobalAveragePooling2D(name="gap")(feature_map)
x = layers.Dropout(0.30, name="head_dropout")(x)
logits = layers.Dense(len(class_names), name="logits")(x)
outputs = layers.Softmax(name="probabilities")(logits)
model = tf.keras.Model(inputs, outputs, name="plantdoc_mobilenetv2")

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        label_smoothing=LABEL_SMOOTHING
    ),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)
model.summary()


## 6. Latih dan pilih checkpoint terbaik

Tahap A melatih classification head dengan backbone dibekukan. Tahap B membuka 40 lapisan terakhir backbone. Kedua tahap menyimpan checkpoint dengan `val_loss` terendah, menggunakan early stopping dan pengurangan learning rate otomatis.

Checkpoint final untuk produksi dipilih dari tahap dengan `val_loss` terbaik, bukan sekadar epoch terakhir. Kedua checkpoint mentah dipertahankan sebagai `best_head.keras` dan `best_finetune.keras` agar baseline serta fine-tuning dapat dibandingkan pada test set yang sama tanpa training ulang.


In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np

OUT = Path("/content/plantdoc_model")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

for name in ("dataset_manifest.json", "split_manifest.csv"):
    shutil.copy2(ROOT / name, OUT / name)


def training_callbacks(checkpoint_name, history_name):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            str(OUT / checkpoint_name),
            monitor="val_loss",
            save_best_only=True,
            mode="min",
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=3,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.3,
            patience=2,
            min_lr=1e-7,
        ),
        tf.keras.callbacks.CSVLogger(str(OUT / history_name)),
        tf.keras.callbacks.TerminateOnNaN(),
    ]


print("=== Tahap A: classification head ===")
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weight,
    callbacks=training_callbacks("best_head.keras", "head_history.csv"),
)

print("=== Tahap B: fine-tuning lapisan terakhir backbone ===")
backbone.trainable = True
for layer in backbone.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        label_smoothing=LABEL_SMOOTHING
    ),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)
history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    class_weight=class_weight,
    callbacks=training_callbacks("best_finetune.keras", "finetune_history.csv"),
)

head_best_epoch = int(np.argmin(history_head.history["val_loss"]))
ft_best_epoch = int(np.argmin(history_ft.history["val_loss"]))
head_best_loss = float(history_head.history["val_loss"][head_best_epoch])
ft_best_loss = float(history_ft.history["val_loss"][ft_best_epoch])
if head_best_loss <= ft_best_loss:
    selected_stage = "head"
    selected_epoch = head_best_epoch + 1
    selected_checkpoint = OUT / "best_head.keras"
else:
    selected_stage = "fine_tune"
    selected_epoch = ft_best_epoch + 1
    selected_checkpoint = OUT / "best_finetune.keras"
selected_val_loss = min(head_best_loss, ft_best_loss)

shutil.copy2(selected_checkpoint, OUT / "best.keras")
(OUT / "training_history.json").write_text(
    json.dumps(
        {
            "head": history_head.history,
            "fine_tune": history_ft.history,
            "selected_stage": selected_stage,
            "selected_epoch": selected_epoch,
            "selected_val_loss": selected_val_loss,
        },
        indent=2,
    ),
    encoding="utf-8",
)
(OUT / "training_config.json").write_text(
    json.dumps(
        {
            "seed": SEED,
            "input_size": list(IMG_SIZE),
            "batch_size": BATCH,
            "epochs_head": EPOCHS_HEAD,
            "epochs_fine_tune": EPOCHS_FT,
            "fine_tune_layers": FINE_TUNE_LAYERS,
            "label_smoothing": LABEL_SMOOTHING,
            "monitor": "val_loss",
            "class_weight": class_weight,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(
    f"Checkpoint dipilih: {selected_stage}, epoch {selected_epoch}, "
    f"val_loss={selected_val_loss:.6f}"
)
print("Checkpoint baseline dan fine-tuning dipertahankan untuk perbandingan.")


## 7. Temperature scaling dari validation set

Model sering terlalu percaya diri pada data baru. Suhu `T` dicari dengan meminimalkan negative log-likelihood **hanya pada validation set**, lalu diterapkan ke bobot dan bias lapisan logits. Dengan cara ini argmax tidak berubah, tetapi probabilitas menjadi lebih terkalibrasi dan ikut diekspor ke TFLite.

Test set tidak digunakan untuk menyetel suhu. Jika sel dijalankan ulang tanpa menghapus `OUT`, suhu yang sudah tersimpan akan dipakai ulang agar kalibrasi tidak diterapkan dua kali.


In [ ]:
from scipy.optimize import minimize_scalar

best = tf.keras.models.load_model(OUT / "best.keras")
y_val = np.concatenate([labels.numpy() for _, labels in val_ds])
val_probabilities = best.predict(val_ds, verbose=0)
log_probabilities = np.log(np.clip(val_probabilities, 1e-7, 1.0))
calibration_path = OUT / "temperature_scaling.json"

if calibration_path.exists():
    calibration = json.loads(calibration_path.read_text(encoding="utf-8"))
    temperature = float(calibration["temperature"])
    print(f"Memakai suhu tersimpan: T={temperature:.6f}")
else:
    def validation_nll(log_temperature):
        candidate = np.exp(log_temperature)
        scaled = log_probabilities / candidate
        return float(-np.mean(scaled[np.arange(len(y_val)), y_val]))

    result = minimize_scalar(
        validation_nll,
        bounds=(np.log(0.25), np.log(5.0)),
        method="bounded",
        options={"xatol": 1e-6},
    )
    if not result.success:
        raise RuntimeError(f"Temperature scaling gagal: {result.message}")
    temperature = float(np.exp(result.x))
    logits_layer = best.get_layer("logits")
    weights, bias = logits_layer.get_weights()
    logits_layer.set_weights([weights / temperature, bias / temperature])
    best.save(OUT / "best.keras")
    scaled_probabilities = np.exp(log_probabilities / temperature)
    scaled_probabilities /= scaled_probabilities.sum(axis=1, keepdims=True)
    unchanged_argmax = bool(
        np.array_equal(
            np.argmax(val_probabilities, axis=1),
            np.argmax(scaled_probabilities, axis=1),
        )
    )
    calibration = {
        "method": "temperature_scaling",
        "fit_split": "val",
        "temperature": temperature,
        "validation_nll_before": validation_nll(0.0),
        "validation_nll_after": validation_nll(result.x),
        "argmax_unchanged": unchanged_argmax,
        "bounds": [0.25, 5.0],
    }
    calibration_path.write_text(
        json.dumps(calibration, indent=2), encoding="utf-8"
    )
    print(json.dumps(calibration, indent=2))

assert calibration["argmax_unchanged"] if "argmax_unchanged" in calibration else True


## 8. Evaluasi held-out dan kalibrasi

Test set tidak dipakai untuk training, early stopping, pemilihan tahap, atau temperature scaling. Metrik utama: accuracy, balanced accuracy, macro precision/recall/F1, F1 khusus penyakit, top-2 accuracy, unknown recall/precision, tomato false rejection rate, healthy-to-disease rate, serta Expected Calibration Error (ECE).


In [ ]:
import csv
from datetime import datetime, timezone

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)


def expected_calibration_error(y_true, y_prob, n_bins=15):
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    correct = predictions == y_true
    bin_ids = np.minimum(
        (confidences * n_bins).astype(int), n_bins - 1
    )
    ece = 0.0
    for bin_id in range(n_bins):
        mask = bin_ids == bin_id
        if not np.any(mask):
            continue
        ece += float(np.mean(mask)) * abs(
            float(np.mean(correct[mask])) - float(np.mean(confidences[mask]))
        )
    return ece


best = tf.keras.models.load_model(OUT / "best.keras")
assert test_ds.class_names == class_names, "Urutan kelas test berbeda dari training"
test_metrics = best.evaluate(test_ds, verbose=0, return_dict=True)
y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_prob = best.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
unknown_index = web_keys.index("unknown")
tomato_indices = [index for index, key in enumerate(web_keys) if key != "unknown"]
disease_indices = [
    index for index, key in enumerate(web_keys) if key in {"early", "late", "mold"}
]
healthy_index = web_keys.index("healthy")
unknown_true = y_true == unknown_index
tomato_true = np.isin(y_true, tomato_indices)
healthy_true = y_true == healthy_index
unknown_recall = (
    float(np.mean(y_pred[unknown_true] == unknown_index))
    if np.any(unknown_true)
    else None
)
tomato_false_rejection_rate = (
    float(np.mean(y_pred[tomato_true] == unknown_index))
    if np.any(tomato_true)
    else None
)
healthy_to_disease_rate = (
    float(np.mean(np.isin(y_pred[healthy_true], disease_indices)))
    if np.any(healthy_true)
    else None
)
disease_true = np.isin(y_true, disease_indices)
disease_macro_f1 = float(
    np.mean([report[class_names[index]]["f1-score"] for index in disease_indices])
)
top2_accuracy = float(
    np.mean(
        [
            true_index in np.argsort(probabilities)[-2:]
            for true_index, probabilities in zip(y_true, y_prob)
        ]
    )
)
ece = expected_calibration_error(y_true, y_prob)

evaluation = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "split": "test",
    "samples": int(len(y_true)),
    "classes": class_names,
    "web_keys": web_keys,
    "loss": float(test_metrics["loss"]),
    "accuracy": float(test_metrics["accuracy"]),
    "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    "macro_precision": float(report["macro avg"]["precision"]),
    "macro_recall": float(report["macro avg"]["recall"]),
    "macro_f1": float(report["macro avg"]["f1-score"]),
    "disease_macro_f1": disease_macro_f1,
    "top2_accuracy": top2_accuracy,
    "expected_calibration_error": ece,
    "unknown_recall": unknown_recall,
    "unknown_precision": float(report["unknown"]["precision"]),
    "unknown_f1": float(report["unknown"]["f1-score"]),
    "tomato_false_rejection_rate": tomato_false_rejection_rate,
    "healthy_to_disease_rate": healthy_to_disease_rate,
    "temperature": temperature,
    "selected_stage": selected_stage,
    "selected_epoch": selected_epoch,
    "selected_val_loss": float(selected_val_loss),
}
(OUT / "evaluation.json").write_text(
    json.dumps(evaluation, indent=2, ensure_ascii=False), encoding="utf-8"
)
(OUT / "classification_report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)

with (OUT / "confusion_matrix.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["actual"] + class_names)
    for name, row in zip(class_names, cm):
        writer.writerow([name, *row.tolist()])

with (OUT / "predictions.csv").open("w", newline="", encoding="utf-8") as handle:
    fieldnames = ["path", "true_class", "predicted_class", *class_names]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for index, (true_index, pred_index, probabilities) in enumerate(
        zip(y_true, y_pred, y_prob)
    ):
        path = Path(test_ds.file_paths[index])
        writer.writerow(
            {
                "path": str(path.relative_to(ROOT)),
                "true_class": class_names[int(true_index)],
                "predicted_class": class_names[int(pred_index)],
                **{
                    name: float(probability)
                    for name, probability in zip(class_names, probabilities)
                },
            }
        )

print(f"Test loss: {test_metrics['loss']:.6f}")
print(f"Test accuracy: {test_metrics['accuracy']*100:.2f}%")
print(f"Balanced accuracy: {evaluation['balanced_accuracy']:.4f}")
print(f"Macro F1: {evaluation['macro_f1']:.4f}")
print(f"Disease macro F1: {disease_macro_f1:.4f}")
print(f"Unknown recall: {unknown_recall:.4f}")
print(f"Unknown precision: {evaluation['unknown_precision']:.4f}")
print(f"Tomato false rejection rate: {tomato_false_rejection_rate:.4f}")
print(f"Expected calibration error: {ece:.4f}")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    cmap="Greens", colorbar=False
)
plt.title("Confusion Matrix - Test Set")
plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

confidences = np.max(y_prob, axis=1)
correct = y_pred == y_true
bin_edges = np.linspace(0.0, 1.0, 16)
bin_ids = np.minimum((confidences * 15).astype(int), 14)
bin_confidence = []
bin_accuracy = []
for bin_id in range(15):
    mask = bin_ids == bin_id
    bin_confidence.append(float(np.mean(confidences[mask])) if np.any(mask) else np.nan)
    bin_accuracy.append(float(np.mean(correct[mask])) if np.any(mask) else np.nan)

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "--", color="gray", label="Ideal")
plt.plot(bin_confidence, bin_accuracy, "o-", color="#138a52", label="Model")
plt.xlabel("Confidence")
plt.ylabel("Accuracy")
plt.title(f"Reliability Diagram - ECE {ece:.4f}")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "calibration.png", dpi=160, bbox_inches="tight")
plt.show()


## 9. Baseline versus fine-tuning

Perbandingan menggunakan dua checkpoint dari eksperimen yang sama:

- **Head-only baseline:** backbone MobileNetV2 ImageNet tetap dibekukan; hanya classification head yang dilatih pada Tahap A.
- **Partial fine-tuning:** proses dimulai dari head yang sama, lalu 40 lapisan terakhir backbone dibuka dan dilatih pada Tahap B.

Keduanya dievaluasi ulang pada validation dan test set yang sama. Suhu kalibrasi masing-masing dicari hanya dari validation set agar perbandingan ECE/NLL adil. Test set tidak digunakan untuk memilih pemenang; pemilihan model produksi tetap berdasarkan `val_loss`.

Model ImageNet zero-shot tidak dipakai sebagai baseline karena output 1.000 kelas ImageNet tidak sebanding secara langsung dengan lima kelas daun tomat.

Output: `model_comparison.json`, `model_comparison.csv`, `per_class_comparison.csv`, `model_comparison.png`, serta `training_curves.png`.


In [ ]:
import csv
import json
from datetime import datetime, timezone

import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from sklearn.metrics import balanced_accuracy_score, classification_report


def calibrate_candidate(candidate):
    validation_labels = np.concatenate([labels.numpy() for _, labels in val_ds])
    validation_probabilities = candidate.predict(val_ds, verbose=0)
    log_probabilities = np.log(np.clip(validation_probabilities, 1e-7, 1.0))

    def validation_nll(log_temperature):
        scaled = log_probabilities / np.exp(log_temperature)
        return float(-np.mean(scaled[np.arange(len(validation_labels)), validation_labels]))

    result = minimize_scalar(
        validation_nll,
        bounds=(np.log(0.25), np.log(5.0)),
        method="bounded",
        options={"xatol": 1e-6},
    )
    if not result.success:
        raise RuntimeError(f"Temperature scaling kandidat gagal: {result.message}")
    candidate_temperature = float(np.exp(result.x))
    logits_layer = candidate.get_layer("logits")
    weights, bias = logits_layer.get_weights()
    logits_layer.set_weights([weights / candidate_temperature, bias / candidate_temperature])
    scaled_probabilities = np.exp(log_probabilities / candidate_temperature)
    scaled_probabilities /= scaled_probabilities.sum(axis=1, keepdims=True)
    unchanged = bool(
        np.array_equal(
            np.argmax(validation_probabilities, axis=1),
            np.argmax(scaled_probabilities, axis=1),
        )
    )
    return candidate_temperature, validation_nll(0.0), validation_nll(result.x), unchanged


def evaluate_candidate(stage, checkpoint_path, stage_val_loss):
    candidate = tf.keras.models.load_model(checkpoint_path)
    candidate_temperature, nll_before, nll_after, unchanged = calibrate_candidate(candidate)
    candidate_metrics = candidate.evaluate(test_ds, verbose=0, return_dict=True)
    candidate_true = np.concatenate([labels.numpy() for _, labels in test_ds])
    candidate_probabilities = candidate.predict(test_ds, verbose=0)
    candidate_predictions = np.argmax(candidate_probabilities, axis=1)
    candidate_report = classification_report(
        candidate_true,
        candidate_predictions,
        labels=list(range(len(class_names))),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    unknown_idx = web_keys.index("unknown")
    tomato_idx = [index for index, key in enumerate(web_keys) if key != "unknown"]
    disease_idx = [
        index for index, key in enumerate(web_keys) if key in {"early", "late", "mold"}
    ]
    unknown_mask = candidate_true == unknown_idx
    tomato_mask = np.isin(candidate_true, tomato_idx)
    summary = {
        "stage": stage,
        "checkpoint": checkpoint_path.name,
        "validation_loss": float(stage_val_loss),
        "temperature": candidate_temperature,
        "validation_nll_before": float(nll_before),
        "validation_nll_after": float(nll_after),
        "calibration_argmax_unchanged": unchanged,
        "trainable_parameters": int(
            sum(np.prod(weights.shape) for weights in candidate.trainable_weights)
        ),
        "total_parameters": int(
            sum(np.prod(weights.shape) for weights in candidate.weights)
        ),
        "test_loss": float(candidate_metrics["loss"]),
        "accuracy": float(candidate_metrics["accuracy"]),
        "balanced_accuracy": float(
            balanced_accuracy_score(candidate_true, candidate_predictions)
        ),
        "macro_precision": float(candidate_report["macro avg"]["precision"]),
        "macro_recall": float(candidate_report["macro avg"]["recall"]),
        "macro_f1": float(candidate_report["macro avg"]["f1-score"]),
        "disease_macro_f1": float(
            np.mean(
                [candidate_report[class_names[index]]["f1-score"] for index in disease_idx]
            )
        ),
        "unknown_recall": (
            float(np.mean(candidate_predictions[unknown_mask] == unknown_idx))
            if np.any(unknown_mask)
            else None
        ),
        "unknown_precision": float(candidate_report["unknown"]["precision"]),
        "unknown_f1": float(candidate_report["unknown"]["f1-score"]),
        "tomato_false_rejection_rate": (
            float(np.mean(candidate_predictions[tomato_mask] == unknown_idx))
            if np.any(tomato_mask)
            else None
        ),
        "expected_calibration_error": expected_calibration_error(
            candidate_true, candidate_probabilities
        ),
    }
    return summary, candidate_report


head_summary, head_report = evaluate_candidate(
    "head_only_baseline", OUT / "best_head.keras", head_best_loss
)
finetune_summary, finetune_report = evaluate_candidate(
    "partial_finetune", OUT / "best_finetune.keras", ft_best_loss
)

comparison_metrics = [
    "validation_loss",
    "test_loss",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "disease_macro_f1",
    "unknown_recall",
    "unknown_precision",
    "tomato_false_rejection_rate",
    "expected_calibration_error",
]
delta = {}
for metric in comparison_metrics:
    baseline_value = head_summary[metric]
    finetune_value = finetune_summary[metric]
    delta[f"finetune_minus_baseline_{metric}"] = (
        float(finetune_value - baseline_value)
        if baseline_value is not None and finetune_value is not None
        else None
    )

comparison = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "test_samples": int(len(y_true)),
    "baseline_definition": "MobileNetV2 backbone frozen; classification head trained",
    "finetune_definition": (
        f"Same head training followed by unfreezing the last {FINE_TUNE_LAYERS} backbone layers"
    ),
    "calibration": "Separate temperature scaling fitted on validation split for each model",
    "production_selection_basis": "validation loss only; test metrics are not used for selection",
    "selected_stage": selected_stage,
    "models": {
        "head_only_baseline": head_summary,
        "partial_finetune": finetune_summary,
    },
    "delta_finetune_minus_baseline": delta,
}
(OUT / "model_comparison.json").write_text(
    json.dumps(comparison, indent=2, ensure_ascii=False), encoding="utf-8"
)

csv_fields = [
    "stage",
    "checkpoint",
    "validation_loss",
    "temperature",
    "trainable_parameters",
    "total_parameters",
    "test_loss",
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "disease_macro_f1",
    "unknown_recall",
    "unknown_precision",
    "unknown_f1",
    "tomato_false_rejection_rate",
    "expected_calibration_error",
]
with (OUT / "model_comparison.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=csv_fields)
    writer.writeheader()
    writer.writerows(
        {field: summary.get(field) for field in csv_fields}
        for summary in (head_summary, finetune_summary)
    )

with (OUT / "per_class_comparison.csv").open(
    "w", newline="", encoding="utf-8"
) as handle:
    fieldnames = ["stage", "class", "precision", "recall", "f1", "support"]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for stage, class_report in (
        ("head_only_baseline", head_report),
        ("partial_finetune", finetune_report),
    ):
        for class_name in class_names:
            values = class_report[class_name]
            writer.writerow(
                {
                    "stage": stage,
                    "class": class_name,
                    "precision": values["precision"],
                    "recall": values["recall"],
                    "f1": values["f1-score"],
                    "support": values["support"],
                }
            )

metric_labels = ["Accuracy", "Macro F1", "Disease F1"]
metric_keys = ["accuracy", "macro_f1", "disease_macro_f1"]
x = np.arange(len(metric_keys))
width = 0.36
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(
    x - width / 2,
    [head_summary[key] for key in metric_keys],
    width,
    label="Head-only baseline",
    color="#718096",
)
axes[0].bar(
    x + width / 2,
    [finetune_summary[key] for key in metric_keys],
    width,
    label="Partial fine-tuning",
    color="#138a52",
)
axes[0].set_xticks(x, metric_labels)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Score")
axes[0].set_title("Perbandingan pada test set")
axes[0].legend()
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.3f", fontsize=8, padding=2)

axes[1].plot(
    range(1, len(history_head.history["val_loss"]) + 1),
    history_head.history["val_loss"],
    "o-",
    label="Head-only val_loss",
    color="#718096",
)
axes[1].plot(
    range(1, len(history_ft.history["val_loss"]) + 1),
    history_ft.history["val_loss"],
    "o-",
    label="Fine-tuning val_loss",
    color="#138a52",
)
axes[1].set_xlabel("Epoch tahap")
axes[1].set_ylabel("Validation loss")
axes[1].set_title("Kurva validation selama training")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT / "model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(
    range(1, len(history_head.history["loss"]) + 1),
    history_head.history["loss"],
    "o-",
    label="Head train loss",
)
axes[0].plot(
    range(1, len(history_head.history["val_loss"]) + 1),
    history_head.history["val_loss"],
    "o--",
    label="Head val loss",
)
axes[0].plot(
    range(1, len(history_ft.history["loss"]) + 1),
    history_ft.history["loss"],
    "s-",
    label="Fine-tune train loss",
)
axes[0].plot(
    range(1, len(history_ft.history["val_loss"]) + 1),
    history_ft.history["val_loss"],
    "s--",
    label="Fine-tune val loss",
)
axes[0].set_title("Loss per tahap")
axes[0].set_xlabel("Epoch tahap")
axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8)
axes[1].plot(
    range(1, len(history_head.history["accuracy"]) + 1),
    history_head.history["accuracy"],
    "o-",
    label="Head train accuracy",
)
axes[1].plot(
    range(1, len(history_head.history["val_accuracy"]) + 1),
    history_head.history["val_accuracy"],
    "o--",
    label="Head val accuracy",
)
axes[1].plot(
    range(1, len(history_ft.history["accuracy"]) + 1),
    history_ft.history["accuracy"],
    "s-",
    label="Fine-tune train accuracy",
)
axes[1].plot(
    range(1, len(history_ft.history["val_accuracy"]) + 1),
    history_ft.history["val_accuracy"],
    "s--",
    label="Fine-tune val accuracy",
)
axes[1].set_title("Accuracy per tahap")
axes[1].set_xlabel("Epoch tahap")
axes[1].set_ylabel("Accuracy")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "training_curves.png", dpi=160, bbox_inches="tight")
plt.show()

for summary in (head_summary, finetune_summary):
    print(
        f"{summary['stage']}: accuracy={summary['accuracy']:.4f}, "
        f"macro_f1={summary['macro_f1']:.4f}, "
        f"disease_f1={summary['disease_macro_f1']:.4f}, "
        f"unknown_recall={summary['unknown_recall']:.4f}, "
        f"ECE={summary['expected_calibration_error']:.4f}, "
        f"T={summary['temperature']:.4f}"
    )
print("Delta fine-tuning minus baseline:")
for metric in comparison_metrics:
    value = delta[f"finetune_minus_baseline_{metric}"]
    print(f" - {metric}: {'n/a' if value is None else f'{value:+.6f}'}")
print("Model produksi tetap dipilih dari validation loss:", selected_stage)


### Saran eksperimen untuk menaikkan akurasi

1. **Audit near-duplicate dan domain gap.** SHA-256 hanya membuang duplikat identik. Tambahkan data foto lapangan dengan pencahayaan dan latar berbeda agar model tidak hanya unggul pada foto PlantVillage.
2. **Uji satu perubahan setiap kali.** Coba resolusi 256×256, AdamW dengan weight decay kecil, atau cosine learning-rate decay sebagai eksperimen terpisah; jangan mengubah semuanya sekaligus.
3. **Ulangi beberapa seed.** Jalankan minimal tiga seed dan laporkan rata-rata plus standar deviasi accuracy/macro F1 agar kenaikan bukan kebetulan split.
4. **Analisis error, bukan hanya akurasi.** Gunakan `predictions.csv`, `per_class_comparison.csv`, dan confusion matrix untuk melihat kelas yang masih tertukar.
5. **Jangan memaksakan fine-tuning.** Jika validation loss fine-tuning lebih buruk, baseline head-only bisa jadi model produksi yang lebih baik.
6. **Jangan menyetel hiperparameter dari test set.** Test hanya untuk pelaporan akhir; semua keputusan harus berdasarkan validation.


## 10. Grad-CAM dan kotak perkiraan area respons penyakit

Grad-CAM memakai gradien skor kelas terhadap feature map konvolusi terakhir MobileNetV2. Heatmap dinormalisasi, di-upsample ke 224×224, lalu ambang batas pada persentil ke-80. Komponen terbesar dijadikan kotak dengan padding 12%.

Kotak **hanya** dibuat bila prediksi teratas termasuk `early`, `late`, atau `mold` dan confidence minimal 0.55. Kotak tidak dibuat untuk `healthy`, `unknown`, atau prediksi penyakit yang tidak melewati ambang. Semua gambar contoh berasal dari test set dan tidak mengubah model.

**Batas interpretasi:** area kotak adalah area yang paling memengaruhi skor model, bukan batas lesi yang sudah diverifikasi. Untuk klaim deteksi penyakit yang sesungguhnya, diperlukan anotasi kotak/segmentasi dan evaluasi IoU/mAP pada dataset berlabel.


In [ ]:
from PIL import Image, ImageDraw
from scipy import ndimage as ndi


DISEASE_KEYS = ("early", "late", "mold")


def model_rgb(image_path):
    with Image.open(image_path) as image:
        return image.convert("RGB").resize(IMG_SIZE, Image.Resampling.BILINEAR)


def normalize_heatmap(heatmap):
    heatmap = np.asarray(heatmap, dtype=np.float32)
    heatmap = np.maximum(heatmap, 0.0)
    maximum = float(np.max(heatmap))
    if maximum <= 1e-8:
        return np.zeros_like(heatmap)
    return heatmap / maximum


def largest_component_box(heatmap):
    threshold = max(float(np.quantile(heatmap, CAM_PERCENTILE / 100.0)), 0.08)
    mask = heatmap >= threshold
    if float(np.mean(mask)) < CAM_MIN_REGION_FRACTION:
        return None
    components, count = ndi.label(mask)
    if count == 0:
        return None
    sizes = np.bincount(components.ravel())
    sizes[0] = 0
    component_id = int(np.argmax(sizes))
    ys, xs = np.where(components == component_id)
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    padding = max(1, int(CAM_PADDING * max(x1 - x0, y1 - y0)))
    x0 = max(0, x0 - padding)
    y0 = max(0, y0 - padding)
    x1 = min(IMG_SIZE[0], x1 + padding)
    y1 = min(IMG_SIZE[1], y1 + padding)
    return [x0, y0, x1, y1]


def gradcam_heatmap(cam_model, image_batch, class_index):
    with tf.GradientTape() as tape:
        feature_maps, probabilities = cam_model(image_batch, training=False)
        class_score = probabilities[0, class_index]
    gradients = tape.gradient(class_score, feature_maps)
    if gradients is None:
        raise RuntimeError("Gradien Grad-CAM tidak tersedia")
    channel_weights = tf.reduce_mean(gradients, axis=(0, 1))
    heatmap = tf.reduce_sum(
        tf.nn.relu(feature_maps[0]) * channel_weights[None, None, :], axis=-1
    )
    heatmap = tf.image.resize(heatmap[..., None], IMG_SIZE, method="bilinear")
    return normalize_heatmap(heatmap.numpy()[..., 0])


def heatmap_image(heatmap):
    color_map = plt.get_cmap("jet")
    colored = color_map(heatmap)[..., :3]
    return Image.fromarray(np.uint8(np.clip(colored * 255.0, 0, 255)))


def localization_panel(image, heatmap, box, top_key, confidence):
    panel = image.convert("RGB").copy()
    if box is not None and heatmap is not None:
        panel = Image.blend(panel, heatmap_image(heatmap), alpha=0.42)
        draw = ImageDraw.Draw(panel)
        draw.rectangle(box, outline=(255, 245, 245), width=6)
        draw.rectangle(box, outline=(220, 38, 38), width=3)
        label = f"{top_key} {confidence:.1%} - Grad-CAM estimate"
        text_y = max(2, box[1] - 16)
        if box[1] < 20:
            text_y = min(IMG_SIZE[1] - 16, box[3] + 4)
        draw.rectangle(
            [box[0], text_y, box[0] + 245, text_y + 16],
            fill=(30, 30, 30),
        )
        draw.text((box[0] + 3, text_y + 2), label, fill=(255, 255, 255))
    else:
        draw = ImageDraw.Draw(panel)
        draw.rectangle([4, 4, IMG_SIZE[0] - 4, 24], fill=(30, 30, 30))
        draw.text(
            (8, 7),
            f"No disease box - prediction: {top_key}",
            fill=(255, 255, 255),
        )
    return panel


cam_model = tf.keras.Model(
    best.inputs,
    [best.get_layer("mobilenetv2").output, best.output],
    name="gradcam_model",
)
print("Grad-CAM feature map:", cam_model.output[0].shape)
print("Disease classes:", DISEASE_KEYS)


In [ ]:
localization_dir = OUT / "localization_examples"
localization_dir.mkdir(parents=True, exist_ok=True)
localization_rows = []
preview_rows = []

for true_key in DISEASE_KEYS:
    true_index = web_keys.index(true_key)
    candidates = [
        index
        for index, label_index in enumerate(y_true)
        if int(label_index) == true_index
    ]
    candidates.sort(
        key=lambda index: (
            web_keys[int(y_pred[index])] != true_key,
            web_keys[int(y_pred[index])] not in DISEASE_KEYS,
            index,
        )
    )
    class_dir = localization_dir / true_key
    class_dir.mkdir(parents=True, exist_ok=True)

    for sample_number, index in enumerate(candidates[:LOCALIZATION_SAMPLES_PER_CLASS]):
        pred_index = int(y_pred[index])
        pred_key = web_keys[pred_index]
        confidence = float(y_prob[index, pred_index])
        image = model_rgb(test_ds.file_paths[index])
        image_batch = np.asarray(image, dtype=np.float32)[None, ...]
        heatmap = None
        box = None
        if pred_key in DISEASE_KEYS and confidence >= LOCALIZATION_MIN_CONFIDENCE:
            heatmap = gradcam_heatmap(cam_model, image_batch, pred_index)
            box = largest_component_box(heatmap)
        panel = localization_panel(image, heatmap, box, pred_key, confidence)
        file_name = (
            f"{sample_number + 1:02d}_true-{true_key}_pred-{pred_key}.png"
        )
        panel.save(class_dir / file_name)
        row = {
            "path": str(Path(test_ds.file_paths[index]).relative_to(ROOT)),
            "image": str((class_dir / file_name).relative_to(OUT)),
            "true_class": true_key,
            "predicted_class": pred_key,
            "confidence": confidence,
            "method": "grad_cam",
            "box_x0": box[0] if box else "",
            "box_y0": box[1] if box else "",
            "box_x1": box[2] if box else "",
            "box_y1": box[3] if box else "",
            "ground_truth_box_available": False,
        }
        localization_rows.append(row)
        if sample_number == 0:
            preview_rows.append((true_key, pred_key, confidence, image, heatmap, panel))

with (OUT / "localization_results.csv").open(
    "w", newline="", encoding="utf-8"
) as handle:
    fieldnames = [
        "path",
        "image",
        "true_class",
        "predicted_class",
        "confidence",
        "method",
        "box_x0",
        "box_y0",
        "box_x1",
        "box_y1",
        "ground_truth_box_available",
    ]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(localization_rows)

fig, axes = plt.subplots(len(preview_rows), 3, figsize=(12, 4 * len(preview_rows)))
if len(preview_rows) == 1:
    axes = np.asarray(axes).reshape(1, 3)
for row_index, (true_key, pred_key, confidence, image, heatmap, panel) in enumerate(
    preview_rows
):
    axes[row_index, 0].imshow(image)
    axes[row_index, 0].set_title(f"Input - true: {true_key}")
    axes[row_index, 1].imshow(
        heatmap_image(heatmap) if heatmap is not None else image,
        cmap="jet" if heatmap is not None else None,
    )
    axes[row_index, 1].set_title("Grad-CAM" if heatmap is not None else "No heatmap")
    axes[row_index, 2].imshow(panel)
    axes[row_index, 2].set_title(f"Prediction: {pred_key} {confidence:.1%}")
    for axis in axes[row_index]:
        axis.axis("off")
fig.suptitle("Contoh lokalisasi pada test set - kotak adalah estimasi respons model")
fig.tight_layout()
fig.savefig(OUT / "localization_examples.png", dpi=160, bbox_inches="tight")
plt.show()

localization_report = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "method": "grad_cam",
    "feature_layer": "mobilenetv2",
    "input_size": list(IMG_SIZE),
    "percentile": CAM_PERCENTILE,
    "minimum_region_fraction": CAM_MIN_REGION_FRACTION,
    "padding": CAM_PADDING,
    "minimum_confidence": LOCALIZATION_MIN_CONFIDENCE,
    "disease_classes": list(DISEASE_KEYS),
    "examples": len(localization_rows),
    "boxes_created": sum(bool(row["box_x0"] != "") for row in localization_rows),
    "ground_truth_boxes_available": False,
    "iou_available": False,
    "map_available": False,
    "limitation": (
        "Kotak menunjukkan area respons model, bukan batas lesi terverifikasi. "
        "IoU dan mAP memerlukan anotasi kotak atau segmentasi ground truth."
    ),
}
(OUT / "localization_report.json").write_text(
    json.dumps(localization_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(json.dumps(localization_report, indent=2, ensure_ascii=False))
print("Contoh gambar kotak tersimpan di", localization_dir)


## 11. Ekspor model dan validasi Keras–LiteRT

`model.tflite` tetap float32 agar paritas probabilitas dapat diperiksa. Validasi membandingkan seluruh vektor probabilitas Keras dan LiteRT pada seluruh test set, bukan hanya argmax. Metadata juga mencatat suhu kalibrasi dan status lokalisasi.

`best.keras` diperlukan untuk menjalankan Grad-CAM di notebook. `best_head.keras` dan `best_finetune.keras` dipertahankan sebagai bukti baseline dan fine-tuning. Backend web saat ini hanya memakai `model.tflite` dan `labels.json` untuk klasifikasi; kotak Grad-CAM belum menjadi endpoint web.


In [ ]:
import time

from ai_edge_litert.interpreter import Interpreter

converter = tf.lite.TFLiteConverter.from_keras_model(best)
tflite_model = converter.convert()
(OUT / "model.tflite").write_bytes(tflite_model)

labels = {
    "classes": class_names,
    "web_keys": web_keys,
    "input_size": list(IMG_SIZE),
    "preprocessing": "mobilenet_v2.preprocess_input tersemat di model",
    "temperature": temperature,
    "localization": {
        "method": "grad_cam",
        "runtime": "notebook_only",
        "ground_truth_boxes": False,
    },
}
assert "unknown" in web_keys, "Model hasil training tidak memiliki kelas unknown"
(OUT / "labels.json").write_text(
    json.dumps(labels, indent=2, ensure_ascii=False), encoding="utf-8"
)

interpreter = Interpreter(model_path=str(OUT / "model.tflite"))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
assert len(input_details) == 1 and len(output_details) == 1
input_index = input_details[0]["index"]
output_index = output_details[0]["index"]
assert input_details[0]["shape"].tolist() == [1, 224, 224, 3]
assert input_details[0]["dtype"] == np.float32
assert output_details[0]["shape"].tolist() == [1, len(class_names)]

atol = 1e-4
rtol = 1e-4
sample_count = 0
top1_matches = 0
allclose_matches = 0
max_abs_diff = 0.0
sum_abs_diff = 0.0
for images, _ in test_ds:
    keras_probabilities = best(images, training=False).numpy()
    for image, keras_probabilities_one in zip(images.numpy(), keras_probabilities):
        interpreter.set_tensor(input_index, np.expand_dims(image, axis=0))
        interpreter.invoke()
        lite_probabilities = interpreter.get_tensor(output_index)[0]
        difference = np.abs(keras_probabilities_one - lite_probabilities)
        sample_count += 1
        top1_matches += int(
            np.argmax(keras_probabilities_one) == np.argmax(lite_probabilities)
        )
        allclose_matches += int(
            np.allclose(
                keras_probabilities_one,
                lite_probabilities,
                atol=atol,
                rtol=rtol,
            )
        )
        max_abs_diff = max(max_abs_diff, float(difference.max()))
        sum_abs_diff += float(difference.sum())

assert sample_count == len(y_true), "Paritas tidak mencakup seluruh data test"
parity = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "runtime": "ai-edge-litert",
    "runtime_version": package_version("ai-edge-litert"),
    "samples": sample_count,
    "atol": atol,
    "rtol": rtol,
    "allclose_passed": allclose_matches == sample_count,
    "top1_agreement": top1_matches / sample_count,
    "max_absolute_difference": max_abs_diff,
    "mean_absolute_difference": sum_abs_diff / (sample_count * len(class_names)),
}
assert parity["allclose_passed"], f"Paritas Keras dan LiteRT gagal: {parity}"
assert parity["top1_agreement"] == 1.0, f"Prediksi top-1 berbeda: {parity}"

benchmark_input = next(iter(test_ds))[0][:1].numpy()
for _ in range(5):
    interpreter.set_tensor(input_index, benchmark_input)
    interpreter.invoke()
latencies_ms = []
for _ in range(30):
    started = time.perf_counter()
    interpreter.set_tensor(input_index, benchmark_input)
    interpreter.invoke()
    latencies_ms.append((time.perf_counter() - started) * 1000)
parity["benchmark"] = {
    "iterations": len(latencies_ms),
    "warmup_iterations": 5,
    "median_ms": float(np.median(latencies_ms)),
    "p95_ms": float(np.quantile(latencies_ms, 0.95)),
    "scope": "set_tensor dan invoke; tidak termasuk decode, resize, atau I/O",
}
(OUT / "tflite_parity.json").write_text(
    json.dumps(parity, indent=2), encoding="utf-8"
)

dataset_manifest = json.loads(
    (OUT / "dataset_manifest.json").read_text(encoding="utf-8")
)
metadata = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model": "MobileNetV2",
    "input": {
        "shape": input_details[0]["shape"].tolist(),
        "dtype": np.dtype(input_details[0]["dtype"]).name,
    },
    "output": {
        "shape": output_details[0]["shape"].tolist(),
        "dtype": np.dtype(output_details[0]["dtype"]).name,
    },
    "classes": class_names,
    "web_keys": web_keys,
    "preprocessing": "mobilenet_v2.preprocess_input tersemat di model",
    "temperature": temperature,
    "dataset_id": dataset_manifest["dataset_id"],
    "dataset_revision": dataset_manifest["revision"],
    "unknown_sources": dataset_manifest["unknown_sources"],
    "split_seed": dataset_manifest["seed"],
    "unknown_class": "unknown" in web_keys,
    "rejection_strategy": "learned_unknown",
    "open_set_rejection": False,
    "localization": {
        "method": "grad_cam",
        "runtime": "notebook_only",
        "ground_truth_boxes": False,
        "supervised_detection": False,
    },
}
(OUT / "model_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)

artifact_manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "artifacts": {
        str(path.relative_to(OUT)): {
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in sorted(OUT.rglob("*"))
        if path.is_file() and path.name != "artifact_manifest.json"
    },
}
(OUT / "artifact_manifest.json").write_text(
    json.dumps(artifact_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Paritas Keras vs LiteRT:", json.dumps(parity, indent=2))
print("File di", OUT)
for path in sorted(OUT.glob("*")):
    if path.is_file():
        print(f" - {path.name} ({path.stat().st_size/1024/1024:.2f} MB)")
print("Contoh lokalisasi:", localization_dir)
print(json.dumps(labels, indent=2, ensure_ascii=False))


## 12. Simpan artefak ke Google Drive

Sel ini menyalin seluruh folder hasil, termasuk kedua checkpoint, hasil perbandingan, gambar kotak, dan manifest, lalu memverifikasi hash setiap file. Jalankan setelah ekspor dan lokalisasi selesai.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DEST = Path("/content/drive/MyDrive/plantdoc")
shutil.copytree(OUT, DEST, dirs_exist_ok=True)
for path in OUT.rglob("*"):
    if path.is_file():
        destination = DEST / path.relative_to(OUT)
        assert sha256_file(path) == sha256_file(destination), (
            f"Hash salinan Drive tidak cocok: {path.name}"
        )
print("Semua artefak tersimpan dan hash terverifikasi di", DEST)


## 13. Hubungkan ke web dan rekomendasi arsitektur

Untuk proyek ini, **Python penuh lebih masuk akal**: training tetap Python/TensorFlow/Keras di Colab, sedangkan serving memakai Python/FastHTML + LiteRT tanpa TensorFlow penuh. JavaScript murni tidak otomatis lebih cepat untuk model TFLite ini; inference browser memerlukan konversi model ke TensorFlow.js serta runtime WebGL/WebGPU dan menambah kompleksitas distribusi.

Salin `model.tflite` dan `labels.json` hasil ekspor ke `artifacts/models/`. Backend membaca `web_keys`, sehingga urutan `early`, `healthy`, `late`, `mold`, `unknown` harus tetap benar. Simpan `evaluation.json`, `model_comparison.json`, `per_class_comparison.csv`, `classification_report.json`, `tflite_parity.json`, `localization_report.json`, dan `artifact_manifest.json` sebagai bukti eksperimen.

**Status kotak di web:** notebook sudah membuat dan menyimpan gambar berkotak, tetapi backend saat ini masih melakukan klasifikasi saja. Menambahkan kotak ke web memerlukan jalur inferensi lokalisasi terpisah dan overlay gambar yang dirender server; jangan menganggap `localization_examples.png` sebagai output endpoint yang sudah aktif.

Periksa `unknown_recall`, `unknown_precision`, `tomato_false_rejection_rate`, dan paritas sebelum mengganti model produksi. Kelas `unknown` dan kotak Grad-CAM tidak menggantikan evaluasi pada data baru maupun diagnosis ahli tanaman.

Menjalankan web: `python -m pip install -r requirements.txt`, lalu `python main.py`, kemudian buka `http://127.0.0.1:5000`.
